# Notebook 07 — SWaT MAML-AE vs Static (Corrected Core, timeout-proof)

The pivotal within-plant test. SWaT has real temporal structure (moderate task diversity,
silhouette ~0.40) and a **strong** static baseline (~0.83 ROC-AUC), unlike SMAP's ~0.49.
So this is where we find out whether correctly-implemented MAML ever helps in your setting.

Both MAML and the static AE are trained here in the same environment on the same data and
scored on the **identical** windowed attack set, so the comparison is apples-to-apples.

### ▶ Run instructions
1. Create a Kaggle Dataset from the `swat_*` files (normal/attack/labels/tasks/splits) and
   attach it as input. Set **accelerator to GPU**.
2. Use **Save Version → "Save & Run All (Commit)"**. The 30k meta-training is ~1–2 h on GPU
   and finishes in one commit; output persists automatically.
3. If ever interrupted, also attach this notebook's previous output and Commit again —
   startup auto-resumes from the newest checkpoint in any attached input.


## 1 — Imports, paths, data, checkpoint discovery

In [1]:
import os, json, pickle, copy, time, warnings
import numpy as np
import torch, torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
warnings.filterwarnings('ignore')
torch.manual_seed(42); np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

INPUT_PATH  = "/kaggle/input/swat-maml-data"   # <-- fix to your dataset path if different
OUTPUT_PATH = "/kaggle/working"
WORK_MODELS = f"{OUTPUT_PATH}/models"
os.makedirs(WORK_MODELS, exist_ok=True); os.makedirs(f"{OUTPUT_PATH}/results", exist_ok=True)

def _find(name):
    for root,_,files in os.walk("/kaggle/input"):
        if name in files: return os.path.join(root,name)
    return None

Xn = np.load(_find("swat_normal.npy")).astype(np.float32)
Xa = np.load(_find("swat_attack.npy")).astype(np.float32)
ya = np.load(_find("swat_attack_labels.npy"))
tasks  = pickle.load(open(_find("swat_tasks.pkl"),"rb"))
splits = json.load(open(_find("swat_task_splits.json")))
meta_train_tasks = splits['meta_train']; meta_val_tasks = splits['meta_val']
N_FEAT = Xn.shape[1]
print(f"normal {Xn.shape} | attack {Xa.shape} | features {N_FEAT} | regimes: train {len(meta_train_tasks)} val {len(meta_val_tasks)}")

def _all_dirs():
    d=[WORK_MODELS]
    for r,_,_ in os.walk("/kaggle/input"): d.append(r)
    return d
def resolve_ckpt(name):
    best_p,best_s=None,-1
    for d in _all_dirs():
        p=os.path.join(d,name)
        if os.path.exists(p):
            try: s=torch.load(p,map_location="cpu").get("step",0)
            except Exception: s=0
            if s>=best_s: best_s,best_p=s,p
    return best_p,best_s
CKPT_NAME="swat_maml_ckpt.pt"; BEST_NAME="swat_maml_best.pt"; STATIC_NAME="swat_static_ae.pt"; MLP_NAME="swat_mlp_ae.pt"
print("resume scan:", resolve_ckpt(CKPT_NAME)[1])


Device: cuda
normal (49500, 51) | attack (44991, 51) | features 51 | regimes: train 5 val 1
resume scan: -1


## 2 — Model classes (LSTM-AE widened to SWaT features; MLP baseline)

In [2]:
class LSTMEncoder(nn.Module):
    def __init__(self, input_size, hidden1=64, hidden2=32, latent=16):
        super().__init__()
        self.lstm1=nn.LSTM(input_size,hidden1,batch_first=True); self.lstm2=nn.LSTM(hidden1,hidden2,batch_first=True); self.fc=nn.Linear(hidden2,latent)
    def forward(self,x): o,_=self.lstm1(x); _,(h,_)=self.lstm2(o); return self.fc(h.squeeze(0))
class LSTMDecoder(nn.Module):
    def __init__(self, latent=16, hidden1=32, hidden2=64, output_size=51, seq_len=30):
        super().__init__(); self.seq_len=seq_len
        self.lstm1=nn.LSTM(latent,hidden1,batch_first=True); self.lstm2=nn.LSTM(hidden1,hidden2,batch_first=True); self.fc=nn.Linear(hidden2,output_size)
    def forward(self,z): r=z.unsqueeze(1).repeat(1,self.seq_len,1); o,_=self.lstm1(r); o,_=self.lstm2(o); return self.fc(o)
class LSTMAutoencoder(nn.Module):
    def __init__(self, input_size, seq_len=30, hidden1=64, hidden2=32, latent=16):
        super().__init__(); self.encoder=LSTMEncoder(input_size,hidden1,hidden2,latent); self.decoder=LSTMDecoder(latent,hidden2,hidden1,input_size,seq_len)
    def forward(self,x): return self.decoder(self.encoder(x))
    def reconstruction_error(self,x): xh=self.forward(x); return torch.mean((x-xh)**2,dim=(1,2))
class MLPAutoencoder(nn.Module):
    def __init__(self, input_size, seq_len=30, latent=16):
        super().__init__(); self.seq_len,self.input_size=seq_len,input_size; flat=seq_len*input_size
        self.encoder=nn.Sequential(nn.Linear(flat,256),nn.ReLU(),nn.Linear(256,64),nn.ReLU(),nn.Linear(64,latent))
        self.decoder=nn.Sequential(nn.Linear(latent,64),nn.ReLU(),nn.Linear(64,256),nn.ReLU(),nn.Linear(256,flat))
    def forward(self,x): b=x.shape[0]; z=self.encoder(x.reshape(b,-1)); return self.decoder(z).reshape(b,self.seq_len,self.input_size)
    def reconstruction_error(self,x): xh=self.forward(x); return torch.mean((x-xh)**2,dim=(1,2))

W=30; S=10
def make_windows(a): return np.array([a[i:i+W] for i in range(0,len(a)-W+1,S)],dtype=np.float32)
NORMAL_WINDOWS = make_windows(Xn)
_idx=[(i,i+W) for i in range(0,len(Xa)-W+1,S)]
ATTACK_WINDOWS = np.array([Xa[a:b] for a,b in _idx],dtype=np.float32)
ATTACK_YW = np.array([int(ya[a:b].any()) for a,b in _idx])
print(f"normal windows {NORMAL_WINDOWS.shape} | attack windows {ATTACK_WINDOWS.shape} | anomaly {ATTACK_YW.sum()}/{len(ATTACK_YW)}")


normal windows (4948, 30, 51) | attack windows (4497, 30, 51) | anomaly 648/4497


## 3 — Corrected FOMAML core (tasks = behavioural regimes)

In [3]:
criterion=nn.MSELoss()
def sample_episode(regime_key, support_size=20, query_size=20, rng=np.random):
    w=tasks[regime_key]['normal_windows']; idx=rng.permutation(len(w)); need=support_size+query_size
    if len(w)<need:
        s=w[rng.choice(len(w),support_size,replace=True)]; q=w[rng.choice(len(w),query_size,replace=True)]
    else: s=w[idx[:support_size]]; q=w[idx[support_size:need]]
    return torch.tensor(s,dtype=torch.float32).to(DEVICE), torch.tensor(q,dtype=torch.float32).to(DEVICE)
def inner_adapt(model, support, inner_lr=0.01, inner_steps=10):
    learner=copy.deepcopy(model); learner.train(); opt=torch.optim.SGD(learner.parameters(),lr=inner_lr)
    for _ in range(inner_steps):
        opt.zero_grad(); loss=criterion(learner(support),support); loss.backward(); opt.step()
    return learner
def outer_step(model, outer_opt, task_batch, inner_lr=0.01, inner_steps=10, support_size=20, query_size=20, train=True, rng=np.random):
    accum=[None]*len(list(model.parameters())); meta_loss=0.0
    for rk in task_batch:
        support,query=sample_episode(rk,support_size,query_size,rng)
        learner=inner_adapt(model,support,inner_lr,inner_steps)
        qloss=criterion(learner(query),query)
        if train:
            g=torch.autograd.grad(qloss,learner.parameters())
            accum=[gi.detach() if a is None else a+gi.detach() for a,gi in zip(accum,g)]
        meta_loss+=qloss.item()
    meta_loss/=len(task_batch)
    if train:
        outer_opt.zero_grad()
        for p,a in zip(model.parameters(),accum): p.grad=a/len(task_batch)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); outer_opt.step()
    return meta_loss
print("core ready")


core ready


## 4 — Meta-train MAML (auto-resume, checkpoint every 500 steps)

In [4]:
INNER_LR=0.01; OUTER_LR=0.001; INNER_STEPS=10; TPB=min(4,len(meta_train_tasks)); SUP=20; QRY=20
N_OUTER=30000; VAL_EVERY=500
WORK_CKPT=f"{WORK_MODELS}/{CKPT_NAME}"; WORK_BEST=f"{WORK_MODELS}/{BEST_NAME}"
rng=np.random.RandomState(42)
model=LSTMAutoencoder(N_FEAT).to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=OUTER_LR)
sched=torch.optim.lr_scheduler.ReduceLROnPlateau(opt,'min',factor=0.5,patience=5,min_lr=1e-5)
start=0; best=float('inf'); hist={'step':[],'train':[],'val':[]}
rp,rs=resolve_ckpt(CKPT_NAME)
if rp is not None and rs>0:
    ck=torch.load(rp,map_location=DEVICE); model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    start=ck['step']; best=ck['best']; hist=ck['hist']; print("RESUMED @", start)
else: print("training from scratch")
def meta_val():
    return float(np.mean([outer_step(model,opt,[c],INNER_LR,INNER_STEPS,SUP,QRY,train=False,rng=rng) for c in meta_val_tasks]))
if start>=N_OUTER: print("already complete; skipping to eval")
else:
    t0=time.time()
    for step in range(start+1,N_OUTER+1):
        batch=rng.choice(meta_train_tasks,size=TPB,replace=False).tolist()
        model.train(); tl=outer_step(model,opt,batch,INNER_LR,INNER_STEPS,SUP,QRY,train=True,rng=rng)
        if step%VAL_EVERY==0:
            vl=meta_val(); sched.step(vl)
            hist['step'].append(step); hist['train'].append(tl); hist['val'].append(vl)
            print(f"step {step:6d} | train {tl:.6f} | val {vl:.6f} | {time.time()-t0:.0f}s")
            torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'step':step,'best':best,'hist':hist}, WORK_CKPT)
            if vl<best: best=vl; torch.save({'model_state_dict':model.state_dict(),'val_loss':vl,'step':step}, WORK_BEST)
    print("done. best val", best); json.dump(hist,open(f"{OUTPUT_PATH}/results/swat_maml_history.json","w"),indent=2)


training from scratch
step    500 | train 0.027952 | val 0.036060 | 138s
step   1000 | train 0.012952 | val 0.031260 | 290s
step   1500 | train 0.012813 | val 0.033279 | 440s
step   2000 | train 0.011053 | val 0.030803 | 587s
step   2500 | train 0.009258 | val 0.023854 | 730s
step   3000 | train 0.007302 | val 0.025092 | 868s
step   3500 | train 0.004639 | val 0.028579 | 1013s
step   4000 | train 0.004573 | val 0.027895 | 1158s
step   4500 | train 0.003335 | val 0.030086 | 1294s
step   5000 | train 0.004021 | val 0.029691 | 1429s
step   5500 | train 0.002626 | val 0.027110 | 1565s
step   6000 | train 0.002116 | val 0.022179 | 1701s
step   6500 | train 0.002666 | val 0.028719 | 1835s
step   7000 | train 0.002320 | val 0.022177 | 1971s
step   7500 | train 0.002257 | val 0.026431 | 2109s
step   8000 | train 0.002453 | val 0.024264 | 2248s
step   8500 | train 0.002398 | val 0.027325 | 2387s
step   9000 | train 0.002724 | val 0.026203 | 2526s
step   9500 | train 0.002099 | val 0.024028 | 26

## 5 — Static AE + MLP baselines (full normal data; skip if saved)

In [5]:
def train_full(model_cls, name):
    path=_find(name) or f"{WORK_MODELS}/{name}"
    m=model_cls(N_FEAT).to(DEVICE)
    if _find(name):
        m.load_state_dict(torch.load(_find(name),map_location=DEVICE)['model_state_dict']); print("loaded",name); return m
    data=torch.tensor(NORMAL_WINDOWS); p=torch.randperm(len(data)); cut=int(0.9*len(data))
    tr=data[p[:cut]]; va=data[p[cut:]].to(DEVICE)
    o=torch.optim.Adam(m.parameters(),lr=1e-3); B=128; best_s=1e9; bad=0; bs=None
    for ep in range(150):
        m.train(); pi=torch.randperm(len(tr))
        for st in range(0,len(tr),B):
            b=tr[pi[st:st+B]].to(DEVICE); o.zero_grad(); l=criterion(m(b),b); l.backward(); o.step()
        m.eval()
        with torch.no_grad(): vl=criterion(m(va),va).item()
        if vl<best_s-1e-6: best_s=vl; bad=0; bs={k:v.clone() for k,v in m.state_dict().items()}
        else:
            bad+=1
            if bad>=15: break
    m.load_state_dict(bs); torch.save({'model_state_dict':m.state_dict(),'val':best_s}, f"{WORK_MODELS}/{name}")
    print(f"{name} trained, val {best_s:.6f}"); return m
static_model=train_full(LSTMAutoencoder, STATIC_NAME)
mlp_model=train_full(MLPAutoencoder, MLP_NAME)


swat_static_ae.pt trained, val 0.002587
swat_mlp_ae.pt trained, val 0.001420


## 6 — Windowed evaluation (identical protocol for all methods)

In [6]:
def metrics_from_scores(scores, yw):
    auc=roc_auc_score(yw,scores); ap=average_precision_score(yw,scores)
    p,r,_=precision_recall_curve(yw,scores); f1=2*p*r/(p+r+1e-12); bi=np.nanargmax(f1)
    sep=scores[yw==1].mean()/scores[yw==0].mean()
    return {'roc_auc':round(float(auc),4),'pr_auc':round(float(ap),4),'best_f1':round(float(f1[bi]),4),
            'best_f1_precision':round(float(p[bi]),4),'best_f1_recall':round(float(r[bi]),4),'separation':round(float(sep),3)}

Wa_t=torch.tensor(ATTACK_WINDOWS).to(DEVICE)
def score_static(m):
    m.eval()
    with torch.no_grad(): return m.reconstruction_error(Wa_t).cpu().numpy()
def score_maml(maml, support_size, seed):
    r=np.random.RandomState(seed)
    sup=torch.tensor(NORMAL_WINDOWS[r.choice(len(NORMAL_WINDOWS),support_size,replace=False)],dtype=torch.float32).to(DEVICE)
    learner=inner_adapt(maml,sup,inner_lr=0.01,inner_steps=10); learner.eval()
    with torch.no_grad(): return learner.reconstruction_error(Wa_t).cpu().numpy()
def score_iso(seed):
    r=np.random.RandomState(seed)
    sf=NORMAL_WINDOWS[r.choice(len(NORMAL_WINDOWS),2000,replace=False)].reshape(2000,-1)
    iso=IsolationForest(n_estimators=100,contamination='auto',random_state=seed).fit(sf)
    return -iso.score_samples(ATTACK_WINDOWS.reshape(len(ATTACK_WINDOWS),-1))

maml_model=LSTMAutoencoder(N_FEAT).to(DEVICE)
maml_model.load_state_dict(torch.load(resolve_ckpt(BEST_NAME)[0],map_location=DEVICE)['model_state_dict'])
print("loaded MAML best")


loaded MAML best


## 7 — Run evaluation: MAML (few-shot) vs Static / MLP / Iso-Forest

In [7]:
SEEDS=[42,123,456,789,1024]; SUPPORT_SIZES=[20,50,100]
results={}
# static + mlp (deterministic, full-data)
results['Static-AE']=metrics_from_scores(score_static(static_model), ATTACK_YW)
results['MLP-AE']=metrics_from_scores(score_static(mlp_model), ATTACK_YW)
# isolation forest (avg over seeds)
iso_aucs=[metrics_from_scores(score_iso(s), ATTACK_YW) for s in SEEDS]
results['Isolation-Forest']={k:round(float(np.mean([d[k] for d in iso_aucs])),4) for k in iso_aucs[0]}
# MAML at each support size (avg over seeds)
for ss in SUPPORT_SIZES:
    runs=[metrics_from_scores(score_maml(maml_model,ss,s), ATTACK_YW) for s in SEEDS]
    results[f'MAML-AE ({ss}-shot)']={k:round(float(np.mean([d[k] for d in runs])),4) for k in runs[0]}
json.dump(results,open(f"{OUTPUT_PATH}/results/swat_results.json","w"),indent=2)
print("evaluation complete")


evaluation complete


## 8 — Results table + the pivotal comparison vs the 0.83 static bar

In [8]:
order=['Static-AE','MLP-AE','Isolation-Forest','MAML-AE (20-shot)','MAML-AE (50-shot)','MAML-AE (100-shot)']
print(f"{'method':>22s} | {'ROC-AUC':>8s} | {'best-F1':>8s} | {'PR-AUC':>7s} | {'separation':>10s}")
print('-'*70)
for m in order:
    r=results[m]; print(f"{m:>22s} | {r['roc_auc']:>8.4f} | {r['best_f1']:>8.4f} | {r['pr_auc']:>7.4f} | {r['separation']:>10.3f}")
print('-'*70)
base=results['Static-AE']['roc_auc']
print(f"\nStatic-AE bar: ROC-AUC {base:.4f}")
for ss in [20,50,100]:
    d=results[f'MAML-AE ({ss}-shot)']['roc_auc']-base
    print(f"  MAML-AE ({ss}-shot) minus Static-AE (ROC-AUC): {d:+.4f}")
print("\nRead: MAML clearing the static bar (positive delta) = meta-learning helps on SWaT.")
print("      MAML at/below the bar = the SMAP null generalises to a richer benchmark too.")


                method |  ROC-AUC |  best-F1 |  PR-AUC | separation
----------------------------------------------------------------------
             Static-AE |   0.8402 |   0.6933 |  0.7329 |      4.559
                MLP-AE |   0.8438 |   0.7074 |  0.7332 |      7.055
      Isolation-Forest |   0.7983 |   0.6871 |  0.6874 |      1.260
     MAML-AE (20-shot) |   0.8170 |   0.6612 |  0.6815 |      6.560
     MAML-AE (50-shot) |   0.8168 |   0.6612 |  0.6813 |      6.526
    MAML-AE (100-shot) |   0.8175 |   0.6612 |  0.6824 |      6.608
----------------------------------------------------------------------

Static-AE bar: ROC-AUC 0.8402
  MAML-AE (20-shot) minus Static-AE (ROC-AUC): -0.0232
  MAML-AE (50-shot) minus Static-AE (ROC-AUC): -0.0234
  MAML-AE (100-shot) minus Static-AE (ROC-AUC): -0.0227

Read: MAML clearing the static bar (positive delta) = meta-learning helps on SWaT.
      MAML at/below the bar = the SMAP null generalises to a richer benchmark too.
